In [1]:
import dolfinx
print(dolfinx.__version__)

[1790446295.674662] [DESKTOP-JTCK1QB:1021 :0]         netlink.c:134  UCX  ERROR   failed to send netlink message on fd=93: Input/output error
[1790446295.674743] [DESKTOP-JTCK1QB:1021 :0]         netlink.c:134  UCX  ERROR   failed to send netlink message on fd=93: Input/output error
0.11.0


In [2]:
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc

from dolfinx import mesh, fem
from dolfinx.fem.petsc import assemble_matrix
from slepc4py import SLEPc

In [3]:
L,B,H = 1.0, 0.5, 0.5
nx,ny,nz = 50,25,25

domain = mesh.create_box(MPI.COMM_WORLD, [np.array([0.0, 0.0, 0.0]), np.array([L,B,H])], [nx,ny,nz], cell_type=mesh.CellType.tetrahedron)

In [4]:
E = 210e9
nu = 0.3
rho = 7800

mu = E / (2.0 * (1.0 + nu))
lmbda = E * nu/ ((1.0 + nu)*(1.0 - 2.0*nu))

In [5]:
V = fem.functionspace(domain,("Lagrange",1,(domain.geometry.dim,)))
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)


def epsilon(u):
    return ufl.sym(ufl.grad(u))


def sigma(u):
    return (
        2.0 * mu * epsilon(u)
        + lmbda * ufl.tr(epsilon(u)) * ufl.Identity(3)
    )

a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx
m = rho * ufl.inner(u, v) * ufl.dx


In [6]:
fdim = domain.topology.dim - 1

fixed_facets = mesh.locate_entities_boundary(
    domain,
    fdim,
    lambda x: np.isclose(x[0], 0.0)
)

fixed_dofs = fem.locate_dofs_topological(
    V,
    fdim,
    fixed_facets
)

u_zero = fem.Constant(
    domain,
    np.zeros(3, dtype=PETSc.ScalarType)
)

bc = fem.dirichletbc(
    u_zero,
    fixed_dofs,
    V
)


In [7]:
A = fem.form(a)
M_form = fem.form(m)

K = assemble_matrix(A, bcs=[bc])
K.assemble()

M = assemble_matrix(M_form, bcs=[bc])
M.assemble()

In [ ]:
eps = SLEPc.EPS().create(domain.comm)

eps.setOperators(K, M)

eps.setProblemType(SLEPc.EPS.ProblemType.GHEP)

# Number of eigenvalues
eps.setDimensions(nev=10)

# Solve for eigenvalues closest to zero
eps.setWhichEigenpairs(
    SLEPc.EPS.Which.SMALLEST_REAL
)

# Use a suitable eigensolver
eps.setType(SLEPc.EPS.Type.KRYLOVSCHUR)

eps.setTolerances(
    tol=1e-8,
    max_it=1000
)

eps.solve()

In [10]:
num_modes = eps.getConverged()

if domain.comm.rank == 0:
    print("Number of converged modes:", num_modes)

for i in range(min(num_modes, 10)):

    eigenvalue = eps.getEigenvalue(i)

    # lambda = omega^2
    omega = np.sqrt(eigenvalue)

    # Hz
    frequency = omega / (2.0 * np.pi)

    if domain.comm.rank == 0:
        print(
            f"Mode {i+1}: "
            f"lambda = {eigenvalue:.6e}, "
            f"omega = {omega:.6f} rad/s, "
            f"frequency = {frequency:.6f} Hz"
        )

Number of converged modes: 0
